## Import bibliothèque

In [2]:
import pandas as pd
import os

## Chargement de données

In [3]:
RAW_DATA_DIR = os.path.join("..","data", "raw")
df_auth = pd.read_csv(os.path.join(RAW_DATA_DIR, 'authentication_logs.csv'))
df_edr = pd.read_csv(os.path.join(RAW_DATA_DIR, 'edr_alerts.csv'))
df_assets = pd.read_csv(os.path.join(RAW_DATA_DIR, 'assets.csv'))
df_users = pd.read_csv(os.path.join(RAW_DATA_DIR, 'users.csv'))

### 1. Analyse de l'exploitabilité du champ cible (analyst_decision dans EDR)

In [4]:
total_edr = len(df_edr)
missing_decisions = df_edr['analyst_decision'].isnull().sum()
pct_missing = (missing_decisions / total_edr) * 100

print(f"\n[BLOCAGE 1] Champ cible de l'EDR (analyst_decision) :")
print(f"   • Volume total d'alertes : {total_edr}")
print(f"   • Valeurs manquantes (non traitées) : {missing_decisions} ({pct_missing:.1f}%)")
print(" • CONSÉQUENCE : Impossible d'entraîner un modèle supervisé direct sur l'historique complet sans biais massif.")


[BLOCAGE 1] Champ cible de l'EDR (analyst_decision) :
   • Volume total d'alertes : 1929
   • Valeurs manquantes (non traitées) : 234 (12.1%)
 • CONSÉQUENCE : Impossible d'entraîner un modèle supervisé direct sur l'historique complet sans biais massif.


### 2. Analyse des ruptures de référentiels (Shadow IT / Obsolescence)

In [5]:
orphaned_auth_users = set(df_auth['user_id'].dropna()) - set(df_users['user_id'].dropna())
pct_orphan_users = (len(orphaned_auth_users) / df_auth['user_id'].nunique()) * 100

print(f"\n[BLOCAGE 2] Fiabilité des référentiels utilisateurs (users.csv) :")
print(f"   • Utilisateurs actifs dans les logs absents du référentiel RH : {len(orphaned_auth_users)} ({pct_orphan_users:.1f}%)")
print("   * CONSÉQUENCE : Impossibilité de croiser de manière fiable les privilèges ou départements pour scorer le risque utilisateur.")


[BLOCAGE 2] Fiabilité des référentiels utilisateurs (users.csv) :
   • Utilisateurs actifs dans les logs absents du référentiel RH : 6 (1.6%)
   * CONSÉQUENCE : Impossibilité de croiser de manière fiable les privilèges ou départements pour scorer le risque utilisateur.


### 3. Analyse des actifs non répertoriés

In [7]:
orphaned_devices = set(df_auth['device_id'].dropna()) - set(df_assets['device_id'].dropna())
print(f"\n[BLOCAGE 3] Couverture du parc informatique (assets.csv) :")
print(f"   • Machines communiquant sans exister dans assets.csv : {len(orphaned_devices)}")
print("   * CONSÉQUENCE : Présence avérée de 'Shadow IT' ou d'actifs obsolètes non critiques aux yeux de l'infrastructure.")


[BLOCAGE 3] Couverture du parc informatique (assets.csv) :
   • Machines communiquant sans exister dans assets.csv : 8
   * CONSÉQUENCE : Présence avérée de 'Shadow IT' ou d'actifs obsolètes non critiques aux yeux de l'infrastructure.


* *Shadow IT : c'est lorsque des employés utilisent des outils informatiques pour leur travail sans que le service informatique de l'entreprise (la DSI) ne soit au courant et ne les ait autorisés.*